# DESC 3x2pt Static-Probes Figure of Merit - Gaussian-Process emulator variant (01b)


- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-10
- Context: SCOC (Survey Cadence Optimization Committee) - DESC Task Force metrics, restricted to the DESC metrics only (3x2pt, Weak Lensing, Supernovae)
- This notebook: **01b**, a variant of `01_3x2pts_DESC_TaskForce_demo.ipynb` that replaces `StaticProbesFoMEmulatorMetricSimple` (bilinear interpolation, years 1/3/6/10 only) with **`StaticProbesFoMEmulatorMetric`** (Gaussian-Process emulator) - the summary metric actually used by the *official* `rubin_sim.maf.batches.science_radar_batch` "Cosmology" group.
- Simulation analyzed: `/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db`
- Everything else (the `ExgalM5WithCuts` parent metric, the per-year depth cuts, `nside=64`, the non-DD footprint) is unchanged from notebook 01, so the two notebooks can be compared side by side.


## What changes compared to notebook 01

| | `StaticProbesFoMEmulatorMetricSimple` (notebook 01) | `StaticProbesFoMEmulatorMetric` (this notebook) |
|---|---|---|
| Interpolation | Bilinear, on a small per-year (area, depth) grid | Gaussian Process (via `george`), on a single 36-point (area, depth, systematics) grid |
| Years supported | Only 1, 3, 6, 10 (separately calibrated grids) | **Any year** - the model has no `year` argument at all, it only depends on the (area, depth) point computed from whichever cut was applied upstream |
| Systematics | Not modeled | Multiplicative shear bias, photo-z scatter/bias, all with adjustable defaults (`shear_m`, `sigma_z`, `sig_delta_z`, `sig_sigma_z`) |
| Extra dependencies | None | `george` (GP library) and `scikit-learn` (`StandardScaler`) |
| Cost per call | Trivial (one bilinear lookup) | Fits GP hyperparameters from scratch every call (seconds) |
| Used by the official `science_radar_batch`? | No (only available as an option) | **Yes** - this is the metric behind the standard show_maf "Cosmology" pages |

Because it has no `year` argument, this notebook evaluates the GP emulator for **all ten years**, not just 1/3/6/10 as in notebook 01. For a direct comparison, we also re-run the Simple emulator (now that the upstream `RectBivateateSpline` typo has been fixed in this environment - see notebook 01 - it needs no workaround here) alongside the GP one, at every year it supports.


## Simulation and code references
- OpSim run analyzed: `baseline_v5.3.6_10yrs.db` (Rubin baseline v5.3.6, 10-year simulation)
- `rubin_sim.maf` source (main branch, retrieved for this notebook):
  - `rubin_sim/maf/metrics/exgal_m5.py` (`ExgalM5`) and `rubin_sim/maf/metrics/weak_lensing_systematics_metric.py` (`ExgalM5WithCuts`) - unchanged from notebook 01
  - `rubin_sim/maf/maf_contrib/static_probes_fom_summary_metric.py` (`StaticProbesFoMEmulatorMetric`, GP-based - the metric this notebook demonstrates)
  - `rubin_sim/maf/metrics/cosmology_summary_metrics.py` (`StaticProbesFoMEmulatorMetricSimple`, kept here only for comparison)
  - `rubin_sim/maf/batches/science_radar_batch.py` (official "Cosmology" batch definition, function `science_radar_batch` - calls `maf.StaticProbesFoMEmulatorMetric(nside=nside, metric_name="3x2ptFoM")` once per year, with the batch's default `nside=64`)


## Practical note before running

`StaticProbesFoMEmulatorMetric` needs `george` and `scikit-learn`, imported lazily inside its `run()` method. If they are missing, install with `pip install george scikit-learn` (`george` needs a C/C++ compiler to build; on macOS with conda, `conda install -c conda-forge george` is usually the smoothest route).

In [ ]:
import importlib

for _pkg in ("george", "sklearn"):
    try:
        importlib.import_module(_pkg)
        print(f"{_pkg}: OK")
    except ImportError as e:
        print(
            f"{_pkg}: NOT FOUND ({e}). Install it before running Section 5 below "
            f"(e.g. 'pip install {_pkg}' or 'conda install -c conda-forge {_pkg}')."
        )

## 1. Imports

In [ ]:
import os
import time
import inspect
from os.path import splitext, basename

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maps
import rubin_sim.maf.metric_bundles as mb

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

Same OpSim file and output-directory convention as notebook 01, with a dedicated `NB_TAG` so outputs don't collide with notebook 01's.

In [ ]:
opsim_fname = "/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db"
assert os.path.isfile(opsim_fname), f"OpSim database not found: {opsim_fname}"

run_name = splitext(basename(opsim_fname))[0]
print("run_name:", run_name)

In [ ]:
NB_TAG = "3X2PTS"
data_dir = f"data_01b_{NB_TAG}"
figs_dir = f"figs_01b_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

In [ ]:
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. The MAF metric classes used

`ExgalM5WithCuts` (parent, per-pixel) is unchanged from notebook 01. The summary metric is now `StaticProbesFoMEmulatorMetric`; we print its docstring and inspect the hard-coded training grid it interpolates over with the Gaussian Process.

In [ ]:
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import ExgalM5WithCuts
from rubin_sim.maf.metrics.cosmology_summary_metrics import StaticProbesFoMEmulatorMetricSimple
from rubin_sim.maf.maf_contrib.static_probes_fom_summary_metric import StaticProbesFoMEmulatorMetric

print(inspect.getdoc(ExgalM5WithCuts))
print("-" * 80)
print(inspect.getdoc(StaticProbesFoMEmulatorMetric))

In [ ]:
# The 36-point (area, depth, systematics) -> FoM training grid the GP interpolates over,
# and the default systematics values used when they are not explicitly overridden:
_probe = StaticProbesFoMEmulatorMetric()
grid_df = pd.DataFrame(_probe.parameters)
print("Training grid ranges:")
print(grid_df[["area", "depth", "FOM"]].describe().loc[["min", "max"]])
print()
print("Default systematics used by this notebook (same as the official batch):")
print(f"  shear_m      = {_probe.shear_m}")
print(f"  sigma_z      = {_probe.sigma_z}")
print(f"  sig_delta_z  = {_probe.sig_delta_z}")
print(f"  sig_sigma_z  = {_probe.sig_sigma_z}")

## 4. Year-dependent configuration (identical to notebook 01)


In [ ]:
bandpass = "i"
nfilters_needed = 6
lim_ebv = 0.2
nside = 64
offset = 0.1

mag_cuts = {
    1: 24.75 - offset,
    2: 25.12 - offset,
    3: 25.35 - offset,
    4: 25.50 - offset,
    5: 25.62 - offset,
    6: 25.72 - offset,
    7: 25.80 - offset,
    8: 25.87 - offset,
    9: 25.94 - offset,
    10: 26.00 - offset,
}
years = sorted(mag_cuts.keys())

# The Simple bilinear emulator is still only calibrated at these years (for comparison only):
simple_fom_defined_years = [1, 3, 6, 10]
# The GP emulator has no such restriction: it will be evaluated at every year below.

pix_area = hp.nside2pixarea(nside, degrees=True)
print(f"nside={nside} -> pixel area = {pix_area:.4f} deg^2")

## 5. Running the metric chain, year by year

Same structure as notebook 01: per year, build the `ExgalM5WithCuts` Healpix depth map over the non-DD footprint, then attach **both** summary emulators as `summary_metrics` so their outputs can be compared directly:
- `StaticProbesFoMEmulatorMetric(nside=nside, metric_name="3x2ptFoM_GP")` (this notebook's subject, defined at every year);
- `StaticProbesFoMEmulatorMetricSimple(year=year, metric_name="3x2ptFoM_simple")` (notebook 01's metric, for comparison, only meaningful at years 1, 3, 6, 10).

In [ ]:
dustmap = maps.DustMap(nside=nside, interp=False)


def run_3x2pt_year_gp(opsim_fname, run_name, year, out_dir):
    """Run the ExgalM5WithCuts + (GP and Simple) 3x2pt-FoM-emulator MAF chain for one survey year."""
    depth_cut = mag_cuts[year]
    sqlconstraint = "night <= %s and scheduler_note not like 'DD%%'" % (year * 365.25 + 0.5)
    info_label = f"{bandpass} band non-DD year {year}"

    parent_metric = metrics.ExgalM5WithCuts(
        lsst_filter=bandpass,
        n_filters=nfilters_needed,
        extinction_cut=lim_ebv,
        depth_cut=depth_cut,
    )

    summary_metrics = [
        metrics.MeanMetric(),
        metrics.MedianMetric(),
        metrics.RmsMetric(),
        metrics.CountRatioMetric(norm_val=1.0 / pix_area, metric_name="Effective Area (deg)"),
        StaticProbesFoMEmulatorMetric(nside=nside, metric_name="3x2ptFoM_GP"),
        StaticProbesFoMEmulatorMetricSimple(year=year, metric_name="3x2ptFoM_simple"),
    ]

    slicer = slicers.HealpixSlicer(nside=nside, use_cache=False)

    bundle = mb.MetricBundle(
        parent_metric,
        slicer,
        sqlconstraint,
        maps_list=[dustmap],
        run_name=run_name,
        info_label=info_label,
        summary_metrics=summary_metrics,
    )
    bd = mb.make_bundles_dict_from_list([bundle])
    bgroup = mb.MetricBundleGroup(bd, opsim_fname, out_dir=out_dir, results_db=resultsDb)
    bgroup.run_all()
    return bundle

In [ ]:
bundles_by_year = {}
t0 = time.time()
for year in years:
    print("Running year", year, "...")
    bundles_by_year[year] = run_3x2pt_year_gp(opsim_fname, run_name, year, data_dir)
print(f"Done in {time.time() - t0:.1f} s")

## 6. Results table

In [ ]:
rows = []
for year in years:
    sv = bundles_by_year[year].summary_values
    rows.append(
        {
            "year": year,
            "i_depth_cut": mag_cuts[year],
            "effective_area_deg2": sv.get("Effective Area (deg)"),
            "median_depth": sv.get("Median"),
            "fom_3x2pt_GP": sv.get("3x2ptFoM_GP"),
            "fom_3x2pt_simple": sv.get("3x2ptFoM_simple") if year in simple_fom_defined_years else np.nan,
        }
    )
results_df = pd.DataFrame(rows).set_index("year")
results_df

In [ ]:
results_csv = os.path.join(data_dir, f"{run_name}_3x2pt_GP_results_by_year.csv")
results_df.to_csv(results_csv)
print("Saved:", results_csv)

## 7. Healpix maps and histograms of the usable extragalactic depth

The underlying per-pixel map (`ExgalM5WithCuts`) is identical to notebook 01 - only the summary FoM differs - so we just show the year-10 map here for reference; see notebook 01 for the full set.

In [ ]:
def save_bundle_plots(bundle, year, figs_dir, tag_prefix):
    made_plots = bundle.plot(savefig=False)
    saved = []
    for plot_type, fig in made_plots.items():
        if fig is None:
            continue
        base = os.path.join(figs_dir, f"{tag_prefix}_year{year:02d}_{plot_type}")
        fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
        fig.savefig(base + ".pdf", bbox_inches="tight")
        saved.append(base)
        plt.close(fig)
    return saved


saved = save_bundle_plots(bundles_by_year[10], 10, figs_dir, f"{run_name}_ExgalM5WithCuts")
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = bundles_by_year[10].plot(savefig=False)
plt.show()

## 8. GP emulator vs Simple emulator, as a function of survey year

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 8), sharex=True)

axs[0].plot(results_df.index, results_df["effective_area_deg2"], marker="o", color="black")
axs[0].set_ylabel("Effective area\n[deg$^2$]")
axs[0].grid(alpha=0.3)

axs[1].plot(
    results_df.index,
    results_df["fom_3x2pt_GP"],
    marker="o",
    color="crimson",
    label="StaticProbesFoMEmulatorMetric (GP, this notebook)",
)
simple_defined = results_df.loc[results_df.index.isin(simple_fom_defined_years)]
axs[1].plot(
    simple_defined.index,
    simple_defined["fom_3x2pt_simple"],
    marker="s",
    linestyle="none",
    color="steelblue",
    label="StaticProbesFoMEmulatorMetricSimple (notebook 01)",
)
axs[1].set_ylabel("3x2pt static-probes FoM")
axs[1].set_xlabel("Survey year")
axs[1].legend()
axs[1].grid(alpha=0.3)

fig.suptitle(f"DESC 3x2pt FoM: GP vs Simple emulator - {run_name}")
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_3x2pt_GP_vs_Simple_vs_year")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 9. Caveats

- The GP is trained on a 36-point grid spanning `area` in roughly [7600, 19800] deg^2 and `depth` in roughly [24.8, 26.8] mag (see Section 3). Early-year cuts (small effective area, shallower depth) can fall near or outside the edge of this grid, so the GP prediction there is an **extrapolation** and should be read with caution - unlike the Simple emulator, which at least has a year-specific calibration for years 1, 3, 6 and 10 (though it covers no other years at all).
- The GP model itself has **no notion of survey year** - it only ever sees the (area, depth) point computed from whichever cut produced the input map. If two different years happened to yield the same (area, depth), the GP would return the same FoM for both. The year-dependence seen in the plot above comes entirely from how (area, depth) evolve with the depth cut and accumulated visits, not from anything year-specific inside the emulator.
- Systematics (`shear_m`, `sigma_z`, `sig_delta_z`, `sig_sigma_z`) are left at their defaults here, matching the official batch. Unlike the Simple emulator, this metric lets you explore sensitivity to those systematics directly by passing different values to the constructor.
- The GP hyperparameters are refit from scratch on every single call (one `scipy.optimize.minimize` per year here); for large batches this is the dominant cost of this metric, though still far cheaper than `SNNSNMetric` (notebook 03).


## References
- Lochner, M. et al. 2018, "Optimizing LSST Observing Strategy for Dark Energy Science", arXiv:1808.00006 - defines the static-probes (3x2pt) Figure-of-Merit forecast grid used by both emulators.
- Zuntz, J. et al. 2021, "The LSST-DESC 3x2pt Tomography Optimization Challenge", arXiv:2108.13418.
- Bianco, F. B. et al. 2022, ApJS 258, 1 - SCOC context.
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/maf_contrib/static_probes_fom_summary_metric.py`, `rubin_sim/maf/batches/science_radar_batch.py`)
- See also the companion `01_3x2pts_DESC_TaskForce_demo.ipynb` notebook (Simple emulator) for the full Healpix-map / histogram walkthrough of `ExgalM5WithCuts`.
